# 🪷 Dharma Archive Pipeline
Rip → Transcribe → Clean → Archive

Run cells top to bottom. Each section is independent.

---
## Cell 1 — Install Dependencies

In [1]:
!jupyter nbextension enable --py widgetsnbextension --user

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: ok


In [2]:
# Cell 1 — Verify environment (no installs needed)
import sys, torch

print(f"Python:  {sys.version}")
print(f"Env:     {sys.prefix}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Quick import check
import whisperx, anthropic, ipywidgets
print("\n✓ All packages available")

Python:  3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Env:     c:\Users\brian\.conda\envs\whisper_env_gpu
Torch:   2.5.1+cu121
CUDA:    True
GPU:     NVIDIA GeForce RTX 2070 Super with Max-Q Design
VRAM:    8.6 GB

✓ All packages available


---
## Cell 2 — Imports & Config

In [3]:
import os, re, json, sqlite3, subprocess, logging
from pathlib     import Path
from datetime    import datetime
from typing      import TypedDict, Optional
from functools   import reduce
from IPython.display import display, HTML, clear_output

import torch
import ipywidgets as w
import anthropic
from tqdm.notebook import tqdm

# And the export path in Cell 13
out_path = "data/exports/dharma_export.jsonl"
# ── Config ────────────────────────────────────────────────────
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPUTE_TYPE  = 'float16' if DEVICE == 'cuda' else 'int8'
WHISPER_MODEL = 'large-v2'   # or 'medium' / 'small' for less VRAM
BATCH_SIZE    = 16
HF_TOKEN      = os.getenv('HF_TOKEN', '')   # for diarization
DB_PATH       = "db/dharma_archive.db"
AUDIO_OUT_DIR = Path("data/processed")
AUDIO_EXTS    = {'.mp3', '.flac', '.wav', '.m4a', '.aac', '.ogg', '.mp4'}

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger(__name__)

print(f'✓ Device: {DEVICE}  |  Torch: {torch.__version__}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

✓ Device: cuda  |  Torch: 2.5.1+cu121
  GPU: NVIDIA GeForce RTX 2070 Super with Max-Q Design
  VRAM: 8.6 GB


---
## Cell 3 — Data Shapes (TypedDict)

In [4]:
class DiscMeta(TypedDict):
    disc_id:       str
    teacher:       str
    teaching_name: str
    teaching_date: str
    location:      str
    tradition:     str
    series:        str
    disc_number:   int
    source_format: str
    notes:         str

class TrackInfo(TypedDict):
    track_number: int
    filename:     str
    path:         str
    duration_sec: float

class TeachingRecord(TypedDict):
    disc_id:             str
    teacher:             str
    teaching_name:       str
    teaching_date:       str
    location:            str
    tradition:           str
    series:              str
    disc_number:         int
    source_format:       str
    notes:               str
    audio_path:          str
    track_count:         int
    track_listing:       str
    language:            str
    duration_min:        int
    transcript_raw:      str
    transcript_clean:    str
    transcript_segments: str
    track_segments:      str
    summary:             str
    key_teachings:       str
    tibetan_terms:       str
    tags:                str
    speakers:            str
    created_at:          str
    error:               str

def make_record(meta: DiscMeta, tracks=None) -> TeachingRecord:
    return {**meta, 'audio_path': '', 'track_count': len(tracks or []),
            'track_listing': json.dumps(tracks or []), 'language': '',
            'duration_min': 0, 'transcript_raw': '', 'transcript_clean': '',
            'transcript_segments': '[]', 'track_segments': '[]',
            'summary': '', 'key_teachings': '[]', 'tibetan_terms': '[]',
            'tags': '[]', 'speakers': '[]',
            'created_at': datetime.now().isoformat(), 'error': ''}

def set_error(record, error): return {**record, 'error': str(error)}

print('✓ Data shapes defined')

✓ Data shapes defined


---
## Cell 4 — 📝 Metadata Entry Form
Fill in the form for each CD, then click **Add CD**.
When all CDs are added, proceed to Cell 5.

In [6]:
# ── Shared job list — accumulates across Add CD clicks ────────
JOBS: list[tuple[str, DiscMeta]] = []

# ── Widget definitions ────────────────────────────────────────
style   = {'description_width': '140px'}
layout  = w.Layout(width='480px')

w_folder   = w.Text(description='📁 CD Folder *',
                    placeholder='/ripped/disc_01',
                    style=style, layout=layout)
w_teacher  = w.Text(description='👤 Teacher *',
                    placeholder='Tsoknyi Rinpoche',
                    style=style, layout=layout)
w_teaching = w.Text(description='📖 Teaching *',
                    placeholder='Nature of Mind',
                    style=style, layout=layout)
w_date     = w.Text(description='📅 Date',
                    placeholder='2019-08-15  or  Summer 2019',
                    style=style, layout=layout)
w_location = w.Text(description='📍 Location',
                    placeholder='Crestone, Colorado',
                    style=style, layout=layout)
w_tradition= w.Dropdown(description='☸️  Tradition',
                    options=['', 'Nyingma', 'Kagyu', 'Sakya', 'Gelug',
                             'Theravada', 'Zen', 'Mahamudra', 'Other'],
                    style=style, layout=layout)
w_series   = w.Text(description='🗂  Series / Retreat',
                    placeholder='Summer Retreat 2019',
                    style=style, layout=layout)
w_disc_num = w.BoundedIntText(description='💿 Disc #',
                    value=1, min=1, max=999,
                    style=style, layout=w.Layout(width='200px'))
w_notes    = w.Textarea(description='📝 Notes',
                    placeholder='Morning session, Q&A included...',
                    rows=2, style=style, layout=layout)

btn_add    = w.Button(description='＋ Add CD', button_style='primary',
                      icon='plus', layout=w.Layout(width='150px'))
btn_clear  = w.Button(description='Clear Form', button_style='',
                      layout=w.Layout(width='120px'))
out_status = w.Output()

def make_disc_id(teacher, teaching, disc_num):
    t = re.sub(r'\W+', '-', teacher.upper())[:20]
    n = re.sub(r'\W+', '-', teaching.upper())[:20]
    return f'{t}__{n}__D{str(disc_num).zfill(3)}'

def on_add(btn):
    with out_status:
        clear_output()
        folder   = w_folder.value.strip()
        teacher  = w_teacher.value.strip()
        teaching = w_teaching.value.strip()

        # Validate required fields
        if not folder:
            display(HTML('<span style="color:red">✗ CD Folder is required</span>')); return
        if not teacher:
            display(HTML('<span style="color:red">✗ Teacher is required</span>')); return
        if not teaching:
            display(HTML('<span style="color:red">✗ Teaching name is required</span>')); return
        if not Path(folder).exists():
            display(HTML(f'<span style="color:orange">⚠ Folder not found: {folder}</span>')); return
        
        # Folder validation
        folder_path = Path(folder)
        if not folder_path.exists():
            display(HTML(f'<span style="color:red">✗ Folder not found: {folder}</span>')); return

        audio_exts  = {'.mp3', '.flac', '.wav', '.m4a', '.aac', '.ogg', '.mp4'}
        track_files = sorted([f.name for f in folder_path.iterdir()
                              if f.suffix.lower() in audio_exts])
        if not track_files:
            display(HTML(f'<span style="color:orange">⚠ No audio files found in: {folder}</span>')); return

        track_list = ''.join(f'<li style="font-size:11px">{t}</li>' for t in track_files)
        display(HTML(f'<b style="color:green">✓ Found {len(track_files)} track(s):</b>'
                     f'<ul style="margin:4px 0">{track_list}</ul>'))

        meta: DiscMeta = {
            'disc_id':       make_disc_id(teacher, teaching, w_disc_num.value),
            'teacher':       teacher,
            'teaching_name': teaching,
            'teaching_date': w_date.value.strip(),
            'location':      w_location.value.strip(),
            'tradition':     w_tradition.value,
            'series':        w_series.value.strip(),
            'disc_number':   w_disc_num.value,
            'source_format': 'CD',
            'notes':         w_notes.value.strip(),
        }
        JOBS.append((folder, meta))

        # Show queue
        rows = ''.join(
            f'<tr><td>{i+1}</td><td>{m["teacher"]}</td>'
            f'<td>{m["teaching_name"]}</td><td>{m["teaching_date"]}</td>'
            f'<td>{m["location"]}</td><td>{Path(f).name}</td></tr>'
            for i, (f, m) in enumerate(JOBS)
        )
        display(HTML(f'''
            <b>✓ Added ({len(JOBS)} total)</b><br><br>
            <table border="1" cellpadding="4" style="border-collapse:collapse;font-size:12px">
            <tr style="background:#f0f0f0">
                <th>#</th><th>Teacher</th><th>Teaching</th>
                <th>Date</th><th>Location</th><th>Folder</th>
            </tr>{rows}</table>
            <br><i>Scroll down to Cell 5 to run the pipeline.</i>
        '''))

        # Auto-advance disc number
        w_disc_num.value += 1

btn_add.on_click(on_add)

def on_clear(btn):
    for widget in [w_folder, w_teacher, w_teaching,
                   w_date, w_location, w_series, w_notes]:
        widget.value = ''
    w_tradition.value = ''
    w_disc_num.value  = 1
    with out_status:
        clear_output()

btn_clear = w.Button(
    description='Clear Form',
    button_style='warning',
    icon='trash',
    layout=w.Layout(width='130px')
)
btn_clear.on_click(on_clear)

display(w.VBox([
    w.HTML('<h3 style="margin-bottom:8px">💿 Add CD to Queue</h3>'),
    w_folder, w_teacher, w_teaching,
    w_date, w_location, w_tradition,
    w_series, w_disc_num, w_notes,
    w.HBox([btn_add, btn_clear]),
    out_status
]))

---
## Cell 5 — Audio: Discover Tracks & Merge

In [ ]:
def natural_sort_key(s):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', s)]

def get_audio_duration(path):
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', path],
        capture_output=True, text=True
    )
    try:    return float(r.stdout.strip())
    except: return 0.0

def discover_tracks(cd_folder: str) -> list[TrackInfo]:
    files = sorted(
        [f for f in Path(cd_folder).iterdir() if f.suffix.lower() in AUDIO_EXTS],
        key=lambda f: natural_sort_key(f.name)
    )
    if not files:
        raise FileNotFoundError(f'No audio files in {cd_folder}')
    return [
        {'track_number': i, 'filename': f.name,
         'path': str(f), 'duration_sec': get_audio_duration(str(f))}
        for i, f in enumerate(files, 1)
    ]

def compute_track_offsets(tracks):
    offsets, cursor = [], 0.0
    for t in tracks:
        offsets.append({**t, 'start_sec': cursor, 'end_sec': cursor + t['duration_sec']})
        cursor += t['duration_sec']
    return offsets

def merge_tracks(tracks: list[TrackInfo], disc_id: str) -> str:
    AUDIO_OUT_DIR.mkdir(exist_ok=True)
    out = AUDIO_OUT_DIR / f'{disc_id}_merged.mp3'
    if out.exists():
        log.info(f'Reusing merged: {out}'); return str(out)

    concat = AUDIO_OUT_DIR / 'concat_list.txt'
    concat.write_text('\n'.join(f"file '{t['path']}'" for t in tracks))

    r = subprocess.run([
        'ffmpeg', '-f', 'concat', '-safe', '0', '-i', str(concat),
        '-ac', '1', '-ar', '16000', '-b:a', '64k',
        str(out), '-y', '-loglevel', 'error'
    ], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'FFmpeg merge failed: {r.stderr}')

    total = sum(t['duration_sec'] for t in tracks) / 60
    log.info(f'Merged {len(tracks)} tracks → {out} ({total:.1f} min)')
    return str(out)

def assign_segments_to_tracks(segments, track_offsets):
    annotated = []
    for seg in segments:
        mid   = (seg.get('start', 0) + seg.get('end', 0)) / 2
        track = next(
            (t for t in track_offsets if t['start_sec'] <= mid < t['end_sec']),
            track_offsets[-1]
        )
        annotated.append({**seg, 'track_number': track['track_number'],
                           'track_filename': track['filename']})
    return annotated

print('✓ Audio helpers defined')

---
## Cell 6 — Load Whisper Model
> ⚠️ Run once. Takes 1–2 min first time (downloads model weights).

In [ ]:
import whisperx

def load_whisper_model():
    print(f'Loading WhisperX {WHISPER_MODEL} on {DEVICE}...')
    model = whisperx.load_model(
        WHISPER_MODEL, device=DEVICE,
        compute_type=COMPUTE_TYPE, language=None
    )
    print('✓ Model loaded')
    return {'model': model, 'align_cache': {}}

def align_segments(segments, language, audio, bundle):
    if language not in ('en', 'zh', 'ja', 'ko', 'fr', 'de', 'es'):
        return segments
    if language not in bundle['align_cache']:
        am, md = whisperx.load_align_model(language_code=language, device=DEVICE)
        bundle['align_cache'][language] = (am, md)
    am, md = bundle['align_cache'][language]
    return whisperx.align(segments, am, md, audio, DEVICE,
                          return_char_alignments=False)['segments']

def transcribe_audio(audio_path, bundle, track_offsets=None, diarize=False):
    print(f'  Transcribing: {Path(audio_path).name}')
    audio    = whisperx.load_audio(audio_path)
    raw      = bundle['model'].transcribe(audio, batch_size=BATCH_SIZE, print_progress=True)
    lang     = raw.get('language', 'unknown')
    print(f'  Detected language: {lang}')
    segments = align_segments(raw['segments'], lang, audio, bundle)
    speakers = []
    if diarize and HF_TOKEN:
        dm       = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=DEVICE)
        result   = whisperx.assign_word_speakers(dm(audio), {'segments': segments})
        segments = result['segments']
        speakers = list({s.get('speaker','') for s in segments if s.get('speaker')})
    track_segs = assign_segments_to_tracks(segments, track_offsets) if track_offsets else segments
    return {
        'text':           ' '.join(s['text'].strip() for s in segments),
        'language':       lang,
        'segments':       segments,
        'speakers':       speakers,
        'track_segments': track_segs,
    }

def free_whisper(bundle):
    del bundle['model']
    bundle['align_cache'].clear()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print('✓ GPU memory freed')

# Load model now
MODEL_BUNDLE = load_whisper_model()

---
## Cell 7 — LLM Cleaning (Claude Haiku)

In [ ]:
CLEAN_SYSTEM = """You process dharma teaching transcripts from CDs.
Speakers may mix Tibetan, Sanskrit, and English.

Return ONLY valid JSON — no markdown, no preamble:
{
  "cleaned_transcript": "full corrected transcript with paragraph breaks",
  "summary": "3-5 sentence summary",
  "key_teachings": ["point 1", "point 2"],
  "tibetan_terms": ["rigpa", "dzogchen"],
  "language": "english | tibetan | mixed",
  "tags": ["meditation", "mahamudra"],
  "estimated_duration_min": 45
}
Rules: preserve Tibetan/Sanskrit terms, fix speech artifacts,
add paragraph breaks at topic shifts, keep teacher's authentic voice."""

def clean_transcript(raw: str, meta: DiscMeta, track_count: int = 1):
    client  = anthropic.Anthropic()
    prompt  = (
        f"Teacher: {meta['teacher']}\n"
        f"Teaching: {meta['teaching_name']}\n"
        f"Date: {meta['teaching_date']}\n"
        f"Location: {meta['location']}\n"
        f"Tradition: {meta['tradition']}\n"
        f"Tracks: {track_count}\n\n"
        f"RAW TRANSCRIPT:\n{raw[:12000]}"
    )
    try:
        resp = client.messages.create(
            model='claude-haiku-4-5', max_tokens=4096,
            system=CLEAN_SYSTEM,
            messages=[{'role': 'user', 'content': prompt}]
        )
        text = resp.content[0].text.replace('```json','').replace('```','').strip()
        return json.loads(text)
    except Exception as e:
        log.warning(f'LLM failed: {e}')
        return {'cleaned_transcript': raw, 'summary': '', 'key_teachings': [],
                'tibetan_terms': [], 'language': 'unknown', 'tags': [],
                'estimated_duration_min': 0}

print('✓ LLM cleaner defined')

---
## Cell 8 — Database

In [ ]:
COLUMNS = (
    'disc_id', 'teacher', 'teaching_name', 'teaching_date', 'location',
    'tradition', 'series', 'disc_number', 'source_format', 'notes',
    'audio_path', 'track_count', 'track_listing',
    'language', 'duration_min',
    'transcript_raw', 'transcript_clean', 'transcript_segments', 'track_segments',
    'summary', 'key_teachings', 'tibetan_terms', 'tags', 'speakers',
    'created_at', 'error'
)

def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS teachings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {', '.join(f'{c} TEXT' for c in COLUMNS)}, UNIQUE(disc_id)
        )""")
    conn.execute("""
        CREATE VIRTUAL TABLE IF NOT EXISTS teachings_fts USING fts5(
            disc_id, teacher, teaching_name, location, tradition,
            transcript_clean, summary, key_teachings, tibetan_terms,
            content='teachings', content_rowid='id'
        )""")
    conn.commit()
    return conn

def save_record(conn, record):
    ph = ', '.join(f':{c}' for c in COLUMNS)
    conn.execute(
        f"INSERT OR REPLACE INTO teachings ({', '.join(COLUMNS)}) VALUES ({ph})",
        {c: record.get(c, '') for c in COLUMNS}
    )
    conn.execute("""
        INSERT OR REPLACE INTO teachings_fts
        (disc_id, teacher, teaching_name, location, tradition,
         transcript_clean, summary, key_teachings, tibetan_terms)
        VALUES (?,?,?,?,?,?,?,?,?)""",
        (record['disc_id'], record['teacher'], record['teaching_name'],
         record['location'], record['tradition'], record['transcript_clean'],
         record['summary'], record['key_teachings'], record['tibetan_terms'])
    )
    conn.commit()
    return record

DB_CONN = init_db()
print(f'✓ Database ready: {DB_PATH}')

---
## Cell 9 — Pipeline Steps & Composition

In [ ]:
def step_discover_merge(cd_folder, meta):
    def _step(record):
        tracks        = discover_tracks(cd_folder)
        track_offsets = compute_track_offsets(tracks)
        merged        = merge_tracks(tracks, record['disc_id'])
        return {**record, 'audio_path': merged,
                'track_count': len(tracks),
                'track_listing': json.dumps(tracks),
                '_track_offsets': track_offsets}
    return _step

def step_transcribe(bundle, diarize=False):
    def _step(record):
        offsets  = record.get('_track_offsets', [])
        result   = transcribe_audio(record['audio_path'], bundle, offsets, diarize)
        updated  = {**record,
                    'transcript_raw':      result['text'],
                    'language':            result['language'],
                    'transcript_segments': json.dumps(result['segments']),
                    'track_segments':      json.dumps(result['track_segments']),
                    'speakers':            json.dumps(result['speakers'])}
        updated.pop('_track_offsets', None)
        return updated
    return _step

def step_clean(meta):
    def _step(record):
        if not record['transcript_raw'].strip():
            return record
        r = clean_transcript(record['transcript_raw'], meta, record['track_count'])
        return {**record,
                'transcript_clean': r.get('cleaned_transcript', ''),
                'summary':          r.get('summary', ''),
                'key_teachings':    json.dumps(r.get('key_teachings', [])),
                'tibetan_terms':    json.dumps(r.get('tibetan_terms', [])),
                'tags':             json.dumps(r.get('tags', [])),
                'language':         r.get('language', record['language']),
                'duration_min':     r.get('estimated_duration_min', 0)}
    return _step

def step_save_db(conn):
    def _step(record): return save_record(conn, record)
    return _step

def safe_pipe(*fns):
    def _run(record):
        current = record
        for fn in fns:
            try:
                current = fn(current)
            except Exception as e:
                log.error(f"Step failed for {record['disc_id']}: {e}")
                return set_error(current, e)
        return current
    return _run

print('✓ Pipeline steps defined')

---
## Cell 10 — ▶️ Run Pipeline
Uses the jobs added in Cell 4.

In [ ]:
# ── Options ───────────────────────────────────────────────────
DIARIZE  = False   # True = speaker labels (needs HF_TOKEN)
SKIP_LLM = False   # True = skip Claude cleaning (faster first pass)

# ── Skip helper ───────────────────────────────────────────────
def already_processed(disc_id, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    row  = conn.execute(
        "SELECT id FROM teachings WHERE disc_id = ? AND error = ''",
        (disc_id,)
    ).fetchone()
    conn.close()
    return row is not None

# ── Run ───────────────────────────────────────────────────────
if not JOBS:
    print('⚠ No jobs queued. Go to Cell 4 and add CDs first.')
else:
    print(f'Processing {len(JOBS)} CD(s)...\n')
    results = []

    for cd_folder, meta in tqdm(JOBS, desc='CDs'):
        print(f"\n{'─'*50}")
        print(f"  {meta['teacher']} — {meta['teaching_name']}")
        print(f"  {meta['teaching_date']}  |  {meta['location']}")
        print(f"  Folder: {cd_folder}")

        # Skip if already successfully processed
        if already_processed(meta['disc_id']):
            print(f"  ↷ Skipping — already in DB")
            continue

        steps = [
            step_discover_merge(cd_folder, meta),
            step_transcribe(MODEL_BUNDLE, DIARIZE),
        ]
        if not SKIP_LLM:
            steps.append(step_clean(meta))
        steps.append(step_save_db(DB_CONN))

        record = safe_pipe(*steps)(make_record(meta))
        results.append(record)

        if record['error']:
            print(f"  ✗ Error: {record['error']}")
        else:
            print(f"  ✓ Saved  |  Tracks: {record['track_count']}"
                  f"  |  Lang: {record['language']}"
                  f"  |  {record['duration_min']} min")
            if record['summary']:
                print(f"  Summary: {record['summary'][:120]}...")

    ok     = [r for r in results if not r['error']]
    failed = [r for r in results if r['error']]
    print(f"\n{'═'*50}")
    print(f'  ✓ {len(ok)} succeeded   ✗ {len(failed)} failed')
    print(f'  Database: {DB_PATH}')

---
## Cell 11 — 🔍 Search the Archive

In [ ]:
def search_teachings(query, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("""
        SELECT t.disc_id, t.teacher, t.teaching_name, t.teaching_date,
               t.location, t.tradition, t.summary, t.language,
               t.duration_min, t.track_count,
               snippet(teachings_fts, 5, '[', ']', '...', 20) AS excerpt
        FROM teachings_fts
        JOIN teachings t ON teachings_fts.rowid = t.id
        WHERE teachings_fts MATCH ?
        ORDER BY rank LIMIT 20""", (query,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]

# ── Search widget ─────────────────────────────────────────────
w_query  = w.Text(placeholder='e.g. rigpa   or   nature of mind   or   Tsoknyi',
                  layout=w.Layout(width='400px'))
btn_srch = w.Button(description='Search', button_style='info', icon='search')
out_srch = w.Output()

def on_search(btn):
    with out_srch:
        clear_output()
        q    = w_query.value.strip()
        if not q: return
        hits = search_teachings(q)
        if not hits:
            display(HTML('<i>No results found.</i>')); return
        rows = ''.join(f"""
            <tr>
                <td><b>{h['teacher']}</b><br><small>{h['teaching_date']} · {h['location']}</small></td>
                <td>{h['teaching_name']}</td>
                <td>{h['language']}</td>
                <td>{h['track_count']} · {h['duration_min']}min</td>
                <td><small>{h['excerpt']}</small></td>
            </tr>""" for h in hits)
        display(HTML(f"""
            <b>{len(hits)} result(s) for "{q}"</b><br><br>
            <table border="1" cellpadding="5" style="border-collapse:collapse;font-size:12px;width:100%">
            <tr style="background:#e8f4f8">
                <th>Teacher</th><th>Teaching</th><th>Lang</th>
                <th>Tracks·Dur</th><th>Excerpt</th>
            </tr>{rows}</table>"""))

btn_srch.on_click(on_search)
display(w.VBox([
    w.HTML('<h3>🔍 Search Archive</h3>'),
    w.HBox([w_query, btn_srch]),
    out_srch
]))

---
## Cell 12 — 📋 Browse All Teachings

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
rows = conn.execute("""
    SELECT teacher, teaching_name, teaching_date, location,
           tradition, language, track_count, duration_min, summary
    FROM teachings WHERE error = ''
    ORDER BY teacher, teaching_date""").fetchall()
conn.close()

if not rows:
    print('No teachings in archive yet.')
else:
    table_rows = ''.join(f"""
        <tr>
            <td><b>{r['teacher']}</b></td>
            <td>{r['teaching_name']}</td>
            <td>{r['teaching_date']}</td>
            <td>{r['location']}</td>
            <td>{r['tradition']}</td>
            <td>{r['language']}</td>
            <td>{r['track_count']} / {r['duration_min']}m</td>
            <td><small>{(r['summary'] or '')[:80]}{'...' if len(r['summary'] or '') > 80 else ''}</small></td>
        </tr>""" for r in rows)
    display(HTML(f"""
        <b>{len(rows)} teaching(s) in archive</b><br><br>
        <table border="1" cellpadding="5"
               style="border-collapse:collapse;font-size:12px;width:100%">
        <tr style="background:#f0f0f0">
            <th>Teacher</th><th>Teaching</th><th>Date</th><th>Location</th>
            <th>Tradition</th><th>Lang</th><th>Tracks/Dur</th><th>Summary</th>
        </tr>{table_rows}</table>"""))

---
## Cell 13 — 💾 Export JSONL (for RAG / fine-tuning)

In [ ]:
out_path = 'dharma_export.jsonl'
conn     = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
rows     = conn.execute("SELECT * FROM teachings WHERE error = ''").fetchall()
conn.close()

with open(out_path, 'w') as f:
    for row in rows:
        f.write(json.dumps(dict(row)) + '\n')

print(f'✓ Exported {len(rows)} records → {out_path}')

---
## Cell 14 — 🧹 Free GPU Memory
Run when done processing all CDs.

In [ ]:
free_whisper(MODEL_BUNDLE)